In [113]:
import csv
import warnings

import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib as mpl

from typing import List
from scipy import integrate
from datetime import datetime, time, timedelta
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm


warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.display.float_format = '.{:.5f}'.format
sns.reset_defaults()
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Linux Libertine"],
    "font.size": 9,
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{libertine}",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Linux Libertine",
    "mathtext.it": "Linux Libertine:italic",
    "mathtext.bf": "Linux Libertine:bold",
})

FIGSIZE_ACM_SMALL = (3.33, 2.06)
FIGSIZE_ACM_SMALL = (7.00, 2.06)

wearables = [[86, 166, 211], [26, 63, 110], [87, 132, 148], [224, 88, 108], [16, 80, 103]]
wearables_palette = [[r/255, g/255, b/255] for r, g, b in wearables ]
sns.set_palette(wearables_palette)

In [189]:
def get_csv_as_list(filename: str) -> list[list[str]]:
    values = []
    with open(filename) as f:
        reader = csv.reader(f)
        for row in reader:
            row[0] = row[0][20:]
            values.append(row)
    return values[1:]

def parse_csv(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
            return pd.DataFrame(data, columns=["message_id", "t_produced", "t_observed"])
    data = get_csv_as_list(filename)
    return parse(data)

def transform_columns_to_numerics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["message_id"] = pd.to_numeric(df["message_id"])
    df["t_produced"] = pd.to_numeric(df["t_produced"])
    df["t_observed"] = pd.to_numeric(df["t_observed"])
    return df

def reduce_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    reduced_df = df.groupby(["message_id"], as_index=True).agg(
        t_produced=("t_produced", "min"), 
        t_observed=("t_observed", "max")
    )
    return reduced_df


def trim_benchmark(df: pd.DataFrame, ramp_up_s: int = 15, ramp_down_s: int = 5) -> pd.DataFrame:
    df = df.copy()
    start_time = df["t_produced"].min() + ramp_up_s * 1_000_000_000
    end_time = df["t_observed"].max() - ramp_down_s * 1_000_000_000

    trimmed = df[(df["t_produced"] >= start_time) & (df["t_observed"] <= end_time)]
    return trimmed

def median_processing_time(df: pd.DataFrame) -> float:
    df = df.copy()
    df["processing_time"] = df["t_observed"] - df["t_produced"]
    median_pt = df["processing_time"].median()
    return median_pt

def convert_to_sec(nano: float) -> float:
    return nano / (1_000_000_000)

def process(filename: str) -> float:
    csv = parse_csv(filename)
    transformed = transform_columns_to_numerics(csv)
    trimmed = trim_benchmark(transformed)
    reduced = reduce_frame(trimmed)
    return convert_to_sec(median_processing_time(reduced))

def average_median(m1: float, m2: float, m3: float) -> float:
    avg = (m1 + m2 + m3) / 3
    return avg

In [268]:
# Mapper

# 1KB, 1RPS, 60s
median01_1kb = process("measurements/mapper/1kb_1RPS_60s_01.csv")
median02_1kb = process("measurements/mapper/1kb_1RPS_60s_02.csv")
median03_1kb = process("measurements/mapper/1kb_1RPS_60s_03.csv")
print(median01_1kb)
print(median02_1kb)
print(median03_1kb)


# 10 KB, 1RPS, 60s
median01_10kb = process("measurements/mapper/10kb_1RPS_60s_01.csv")
median02_10kb = process("measurements/mapper/10kb_1RPS_60s_02.csv")
median03_10kb = process("measurements/mapper/10kb_1RPS_60s_03.csv")
print(median01_10kb)
print(median02_10kb)
print(median02_10kb)

# 50 KB, 1RPS, 60s
median01_50kb = process("measurements/mapper/50kb_1RPS_60s_01.csv")
median02_50kb = process("measurements/mapper/50kb_1RPS_60s_02.csv")
# median03_100kb = process("measurements/mapper/100kb_1RPS_60s_03.csv")
print(median01_50kb)
print(median02_50kb)
# print(median03_100kb)

print("mapper 1kb:")
print(f"{average_median(median01_1kb, median02_1kb, median03_1kb)}s")
print("mapper 10rps:")
print(f"{average_median(median01_10kb, median02_10kb, median03_10kb)}s")
# print("mapper 100rps:")
# print(f"{average_median(median01_100rps, median02_100rps, median03_100rps)}s")



0.2129881745
0.2106838575
0.211222447
1.555559581
1.5656141035
1.5656141035
20.0448641125
25.116797442
mapper 1kb:
0.21163149299999998s
mapper 10rps:
1.560024951s


In [220]:
# RPS effects on processing time:


plot3r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [34.336, 36.22, 38.333] # TODO -> Test values
})

plot5r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [32.12, 33.5, 35.99] 
})

plot7r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [31.00, 32.8, 35.7] 
})

df = pd.concat([
    plot3r.assign(system="3 Replicas"),
    plot5r.assign(system="5 Replicas"),
    plot7r.assign(system="7 Replicas"),
])

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=df,
    x="Requests/Second",
    y="Median Latency",
    hue="system",
    style="system",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_ylabel("Median Latency [s]")
ax.set_xlabel("Requests/Second")
ax.set_ylim(30, 45)
ax.set_xlim(0.5, 3.5)
ax.set_xticks([1,2,3])
ax.set_xticklabels(["$1$", "$10$", "$100$"])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, loc="center left")

fig.savefig("plots/rps_impact_on_processing_time.pdf", bbox_inches=False)
plt.close()

In [256]:
# Message-Size Impact on Processing Time

plot = pd.DataFrame({
    "MessageSize": [1, 2, 3, 4],
    "ProcessingTime": [0.21163149299999998, 1.560024951, 10, 10],
})

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=plot,
    x="MessageSize",
    y="ProcessingTime",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_ylabel("Processing Time [s]")
ax.set_xlabel("Message Size [KB]")
ax.set_ylim(0, 10)
ax.set_xlim(0.5, 4.5)
ax.set_xticks([1,2,3,4])
ax.set_xticklabels(["$1$", "$10$", "$100$", "$200$"])

fig.savefig("plots/message-size-impact.pdf", bbox_inches=False)
plt.close()